In [0]:
-- Calculate behavioral scores, membership archetypes, and rfm frameworks for both customers and sellers

-- 1. Statistical distribution profiling: rounding them to the nearest "membership milestone"
-- 1.1 customer vip threshold: rounding p95 spend up to the nearest 50
DECLARE OR REPLACE VARIABLE p95_customer_spend DOUBLE;
SET VAR p95_customer_spend = (
  SELECT CEIL(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY total_spend) / 50) * 50 
  FROM (SELECT SUM(total_payment) as total_spend FROM olist_maas_pipeline.gold.fact_orders GROUP BY customer_unique_id)
);

-- 1.2 seller leader threshold: rounding p90 gmv up to the nearest 1000
DECLARE OR REPLACE VARIABLE p90_seller_gmv DOUBLE;
SET VAR p90_seller_gmv = (
  SELECT CEIL(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY total_gmv) / 1000) * 1000 
  FROM (SELECT seller_id, SUM(gmv) as total_gmv FROM olist_maas_pipeline.gold.fact_order_items GROUP BY 1)
);

-- 1.3 seller innovation threshold: rounding p90 sku count up to the nearest 10
DECLARE OR REPLACE VARIABLE p90_sku_diversity DOUBLE;
SET VAR p90_sku_diversity = (
  SELECT CEIL(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY sku_count) / 10) * 10 
  FROM (SELECT seller_id, COUNT(DISTINCT product_id) as sku_count FROM olist_maas_pipeline.gold.fact_order_items GROUP BY 1)
);

-- Threshold audit report
SELECT 
    'Membership VIP Threshold (R$)' AS strategy_metric
    , p95_customer_spend AS current_milestone
UNION ALL
SELECT 
    'Market Leader GMV Floor (R$)'
    , p90_seller_gmv
UNION ALL
SELECT 
    'Innovation SKU Milestone'
    , p90_sku_diversity;

In [0]:
-- 2. Customer scoring & rfm 11-framework: combining strategic goals (archetypes) with relative standing (rfm)
CREATE OR REPLACE TABLE olist_maas_pipeline.gold.dim_customer
USING DELTA
AS
WITH customer_base AS (
    SELECT 
        customer_unique_id
        , FIRST(customer_state) AS customer_state
        , MIN(purchase_date) AS first_order_date
        , MAX(purchase_date) AS last_order_date
        , COUNT(order_id) AS total_orders
        , SUM(total_payment) AS total_spend
        , AVG(review_score) AS avg_rating
        , SUM(CASE WHEN is_on_time = TRUE THEN 1 ELSE 0 END) AS on_time_orders
        , SUM(CASE WHEN is_at_breaking_point = TRUE THEN 1 ELSE 0 END) AS extreme_delay_orders
    FROM olist_maas_pipeline.gold.fact_orders
    GROUP BY customer_unique_id
)
, scores_and_ntiles AS (
    SELECT 
        *
        , DATEDIFF((SELECT MAX(last_order_date) FROM customer_base), last_order_date) AS days_since_last_purchase
        , ROUND(on_time_orders * 1.0 / NULLIF(total_orders, 0), 4) AS on_time_rate
        , LEAST(100, ROUND((total_spend / p95_customer_spend) * 100, 0)) AS ltv_score
        -- rfm quintiles
        , NTILE(5) OVER (ORDER BY DATEDIFF((SELECT MAX(last_order_date) FROM customer_base), last_order_date) DESC) AS r 
        , NTILE(5) OVER (ORDER BY total_orders ASC) AS f
        , NTILE(5) OVER (ORDER BY total_spend ASC) AS m
    FROM customer_base
)
SELECT 
    *
    -- strategic archetype 
    , CASE 
        WHEN total_spend >= p95_customer_spend THEN 'VIP_PLATINUM'
        WHEN total_orders >= 2 THEN 'LOYALIST'
        WHEN days_since_last_purchase > 180 THEN 'HIBERNATING'
        ELSE 'STANDARD'
      END AS customer_archetype
    -- rfm 11-segment framework
    , CASE 
        WHEN (r >= 4 AND f >= 4 AND m >= 4) THEN 'CHAMPIONS'
        WHEN (r >= 2 AND f >= 3 AND m >= 3) THEN 'LOYAL_CUSTOMERS'
        WHEN (r >= 3 AND f <= 3 AND m <= 3) THEN 'POTENTIAL_LOYALISTS'
        WHEN (r >= 4 AND f <= 1) THEN 'NEW_CUSTOMERS'
        WHEN (r >= 3 AND r <= 4 AND f <= 1) THEN 'PROMISING'
        WHEN (r >= 2 AND r <= 3 AND f >= 2 AND f <= 3 AND m >= 2 AND m <= 3) THEN 'NEEDS_ATTENTION'
        WHEN (r >= 2 AND r <= 3 AND f <= 2 AND m <= 2) THEN 'ABOUT_TO_SLEEP'
        WHEN (r <= 2 AND f >= 2 AND m >= 2) THEN 'AT_RISK'
        WHEN (r <= 1 AND f >= 4 AND m >= 4) THEN 'CANT_LOSE_THEM'
        WHEN (r <= 2 AND f <= 2 AND m <= 2) THEN 'HIBERNATING'
        ELSE 'LOST'
      END AS rfm_segment
FROM scores_and_ntiles;

In [0]:
-- 3. Seller scoring & rfm framework: Integrating market leadership milestones with active performance segments
CREATE OR REPLACE TABLE olist_maas_pipeline.gold.dim_seller
USING DELTA
AS
WITH seller_metrics AS (
    SELECT 
        i.seller_id
        , SUM(i.gmv) AS total_gmv
        , COUNT(DISTINCT i.product_id) AS sku_count
        , AVG(o.review_score) AS avg_rating
        , COUNT(DISTINCT i.order_id) AS total_orders
        , MAX(o.purchase_date) AS last_sale_date
        , SUM(CASE WHEN o.is_on_time = TRUE THEN 1 ELSE 0 END) AS on_time_orders
    FROM olist_maas_pipeline.gold.fact_order_items i
    JOIN olist_maas_pipeline.gold.fact_orders o ON i.order_id = o.order_id
    GROUP BY i.seller_id
)
, seller_rfm_logic AS (
    SELECT 
        m.*
        , s.seller_state
        , s.seller_region
        , DATEDIFF((SELECT MAX(last_sale_date) FROM seller_metrics), m.last_sale_date) AS days_since_last_sale
        -- rfm quintiles for sellers
        , NTILE(5) OVER (ORDER BY DATEDIFF((SELECT MAX(last_sale_date) FROM seller_metrics), m.last_sale_date) DESC) AS r 
        , NTILE(5) OVER (ORDER BY m.total_orders ASC) AS f
        , NTILE(5) OVER (ORDER BY m.total_gmv ASC) AS m_rank
    FROM seller_metrics m
    LEFT JOIN olist_maas_pipeline.silver.sales_sellers s ON m.seller_id = s.seller_id
)
SELECT 
    *
    -- strategic label
    , CASE 
        WHEN total_gmv >= p90_seller_gmv AND avg_rating >= 4.2 THEN 'MARKET_LEADER'
        WHEN sku_count >= p90_sku_diversity THEN 'INNOVATOR'
        ELSE 'STABLE_PARTNER'
      END AS strategic_label
    -- rfm 11-segment seller framework
    , CASE 
        WHEN (r >= 4 AND f >= 4 AND m_rank >= 4) THEN 'STRATEGIC_PARTNERS'
        WHEN (r >= 2 AND f >= 3 AND m_rank >= 3) THEN 'CONSISTENT_REVENUE'
        WHEN (r >= 4 AND f <= 2 AND m_rank >= 4) THEN 'RISING_STARS'
        WHEN (r <= 2 AND f >= 3 AND m_rank >= 3) THEN 'AT_RISK_GOLIATHS'
        WHEN (r <= 1 AND f >= 4 AND m_rank >= 4) THEN 'URGENT_WINBACK'
        WHEN (r <= 2 AND f <= 2) THEN 'DORMANT_SMALL_BIZ'
        ELSE 'ACTIVE_MERCHANT'
      END AS performance_segment
    -- reliability tier
    , CASE 
        WHEN (on_time_orders * 1.0 / NULLIF(total_orders, 0)) >= 0.90 THEN 'ELITE_LOGISTICS'
        WHEN (on_time_orders * 1.0 / NULLIF(total_orders, 0)) < 0.70 THEN 'LOGISTICS_CRISIS'
        ELSE 'CONSISTENT'
      END AS logistics_reliability_tier
FROM seller_rfm_logic;